In [1]:
import json
import re
from collections import Counter
import numpy as np
import pandas as pd

PARAMS_PATH = "logreg_params.npz"
TFIDF_META_PATH = "logreg_tfidf.json"


def fill_missing_text_columns(X_df, columns):
    for col in columns:
        X_df[col] = X_df[col].fillna("")


def build_text_specs(tfidf_meta):
    return [
        (
            col,
            {k: int(v) for k, v in tfidf_meta[col]["vocab"].items()},
            np.asarray(tfidf_meta[col]["idf"], dtype=np.float64),
        )
        for col in ["feel_text", "food", "soundtrack"]
    ]


def load_logreg_artifacts(params_path=PARAMS_PATH, tfidf_meta_path=TFIDF_META_PATH):
    params = np.load(params_path, allow_pickle=True)
    with open(tfidf_meta_path, "r", encoding="utf-8") as f:
        tfidf_meta = json.load(f)

    artifacts = {
        "coef": np.asarray(params["coef"], dtype=np.float64),
        "intercept": np.asarray(params["intercept"], dtype=np.float64).ravel(),
        "class_order": params["class_order"].tolist(),
        "numeric_cols": params["numeric_cols"].tolist(),
        "num_means": params["num_means"].astype(np.float64),
        "num_stds": params["num_stds"].astype(np.float64),
        "export_train_indices": np.asarray(params["train_indices"]),
        "export_test_indices": np.asarray(params["test_indices"]),
    }
    artifacts["num_stds"] = np.where(np.abs(artifacts["num_stds"]) < 1e-12, 1.0, artifacts["num_stds"])
    artifacts["text_specs"] = build_text_specs(tfidf_meta)
    return artifacts


df = pd.read_csv("cleaned_dataset.csv")
X = df.drop(["target", "id", "painting"], axis=1).copy()
y = df["target"].copy()

fill_missing_text_columns(X, ["feel_text", "food", "soundtrack"])

artifacts = load_logreg_artifacts()
coef = artifacts["coef"]
intercept = artifacts["intercept"]
class_order = artifacts["class_order"]
numeric_cols = artifacts["numeric_cols"]
num_means = artifacts["num_means"]
num_stds = artifacts["num_stds"]
export_train_indices = artifacts["export_train_indices"]
export_test_indices = artifacts["export_test_indices"]
TEXT_SPECS = artifacts["text_specs"]

In [2]:
TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")


def build_tfidf_block(text_series, vocab, idf_vec):
    block = np.zeros((len(text_series), len(vocab)), dtype=np.float64)

    for i, text in enumerate(text_series):
        tokens = TOKEN_RE.findall(str(text).lower())
        if not tokens:
            continue
        counts = Counter(tok for tok in tokens if tok in vocab)
        if not counts:
            continue

        total = float(sum(counts.values()))
        norm_sq = 0.0

        for tok, cnt in counts.items():
            j = vocab[tok]
            val = (cnt / total) * idf_vec[j]
            block[i, j] = val
            norm_sq += val * val

        norm = np.sqrt(norm_sq)
        if norm > 0.0:
            block[i, :] /= norm

    return block


def build_logreg_features(X_df):
    X_num = X_df[numeric_cols].to_numpy(dtype=np.float64)
    X_num = np.where(np.isnan(X_num), num_means, X_num)
    X_num = (X_num - num_means) / num_stds

    text_blocks = [
        build_tfidf_block(X_df[col], vocab, idf_vec)
        for col, vocab, idf_vec in TEXT_SPECS
    ]

    return np.hstack([X_num] + text_blocks)


def predict_proba(X_mat, coef_mat, intercept_vec):
    if coef_mat.ndim == 1 or (coef_mat.ndim == 2 and coef_mat.shape[0] == 1):
        coef_vec = coef_mat.ravel()
        logits = np.asarray(X_mat @ coef_vec + float(intercept_vec[0])).ravel()
        logits = np.clip(logits, -50.0, 50.0)
        p_pos = 1.0 / (1.0 + np.exp(-logits))
        return np.column_stack([1.0 - p_pos, p_pos])

    logits = np.asarray(X_mat @ coef_mat.T) + intercept_vec.reshape(1, -1)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def infer_model_shape(coef_mat):
    if coef_mat.ndim == 2 and coef_mat.shape[0] == 1:
        return 2, coef_mat.shape[1]
    if coef_mat.ndim == 2:
        return coef_mat.shape
    return 2, coef_mat.shape[0]

In [3]:
def split_by_export_indices(X_df, train_idx, test_idx):
    X_train_df = X_df.loc[train_idx].copy()
    X_test_df = X_df.loc[test_idx].copy()
    return X_train_df, X_test_df


def predict_for_splits(X_train_df, X_test_df, coef_mat, intercept_vec):
    X_train_tx = build_logreg_features(X_train_df)
    X_test_tx = build_logreg_features(X_test_df)
    train_proba = predict_proba(X_train_tx, coef_mat, intercept_vec)
    test_proba = predict_proba(X_test_tx, coef_mat, intercept_vec)
    return X_train_tx, X_test_tx, train_proba, test_proba


X_train, X_test = split_by_export_indices(X, export_train_indices, export_test_indices)
train_indices = X_train.index.to_numpy()
test_indices = X_test.index.to_numpy()

X_train_tx, X_test_tx, logreg_proba_train, logreg_test_proba_test = predict_for_splits(
    X_train, X_test, coef, intercept
)
n_classes, n_features = infer_model_shape(coef)
logreg_class_order = list(class_order)

In [4]:
def build_stack_payload(train_proba, test_proba, class_labels, train_idx, test_idx):
    return {
        "logreg_proba_train": train_proba,
        "logreg_test_proba_test": test_proba,
        "logreg_class_order": class_labels,
        "train_indices": train_idx,
        "test_indices": test_idx,
    }


logreg_stack_payload = build_stack_payload(
    logreg_proba_train,
    logreg_test_proba_test,
    logreg_class_order,
    train_indices,
    test_indices,
)

print("Saved: logreg_branch_for_bernoulli_stack.npz")
pd.DataFrame(logreg_test_proba_test[:5], columns=[f"p_{c}" for c in logreg_class_order])

Saved: logreg_branch_for_bernoulli_stack.npz


,p_0,p_1,p_2
0,0.512514,0.426382,0.061104
1,0.958231,0.038475,0.003294
2,0.006384,0.203622,0.789994
3,0.000008,0.003237,0.996755
4,0.988461,0.011368,0.000171
